# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [98]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [99]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

26


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [100]:
# Download PDF, load text, build prompt, call OpenAI, and parse structured JSON into a Pydantic model
import os, re, json, tempfile, requests
from pydantic import BaseModel
from langchain_community.document_loaders import PyPDFLoader
from openai import OpenAI

# Load prompt templates from file (developer + user_template)
prompt_path = os.path.join('prompts','summary_prompt.txt')
with open(prompt_path, 'r', encoding='utf-8') as f:
    raw = f.read()
parts = raw.split('# user_template')
developer = parts[0].replace('# developer','').strip()
user_template = parts[1].strip()

# Download the PDF to a temporary file


context = '\n\n'.join(p.page_content for p in docs)
# Truncate long context to avoid hitting token limits; you can implement chunking if desired
# max_context_chars 
# if len(context) > max_context_chars:
#     context = context[:max_context_chars] + '\n\n[TRUNCATED]'

# Build user prompt by filling the user template dynamically
tone = 'Formal Academic Writing'
max_summary_tokens = 8000
user_msg = user_template.format(context=context, tone=tone, max_summary_tokens=max_summary_tokens)

# Call OpenAI ChatCompletion (not GPT-5 family) - choose appropriate model available to you

client = OpenAI()


client.api_key = os.getenv('OPENAI_API_KEY')
model = 'gpt-4o-mini'
messages = [
    {'role':'system','content': developer},
]
params = {
        "model": model,
        "input": messages,
        "max_output_tokens": max_summary_tokens,
        "temperature": 0,
        "tools": None,
    }
completion = client.responses.create(**params)

In [101]:
completion.output[0].content[0].text


'```python\nfrom pydantic import BaseModel\n\nclass ArticleSummary(BaseModel):\n    Author: str = "Unknown"\n    Title: str = "Unknown"\n    Relevance: str = "This article is crucial for AI professionals as it delves into the latest advancements and methodologies in artificial intelligence, providing insights that can enhance their skill set and understanding of the field."\n    Summary: str = "The article discusses recent breakthroughs in artificial intelligence, focusing on innovative algorithms and their applications across various industries. It highlights the importance of ethical considerations in AI development and the need for continuous learning to keep pace with rapid technological changes. The author emphasizes collaboration between academia and industry to foster innovation and address challenges in AI deployment."\n    Tone: str = "Formal Academic Writing"\n    InputTokens: int = 0  # Replace with actual input token count\n    OutputTokens: int = 0  # Replace with actual o

In [102]:
completion.output[0].content[0].text.__sizeof__()

1051

In [103]:
context

'pg. 1 \n \n \nThe GenAI Divide  \nSTATE OF AI IN \nBUSINESS 2025 \n \n \n \n \n \n \nMIT NANDA \nAditya Challapally \nChris Pease \nRamesh Raskar \nPradyumna Chari \nJuly 2025\n\npg. 2 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nNOTES \nPreliminary Findings from AI Implementation Research from Project NANDA \nReviewers: Pradyumna Chari, Project NANDA \nResearch Period: January – June 2025 \nMethodology: This report is based on a multi-method research design that includes \na systematic review of over 300 publicly disclosed AI initiatives, structured \ninterviews with representatives from 52 organizations, and survey responses from \n153 senior leaders collected across four major industry conferences. \n Disclaimer: The views expressed in this report are solely those of the authors and \nreviewers and do not reflect the positions of any affiliated employers. \n Confidentiality Note: All company-specific data and quotes have been \nanonymized to maintain compliance with corpora

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [104]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
...

test_case = LLMTestCase(input=context, actual_output=completion.output[0].content[0].text)
metric = SummarizationMetric(
    threshold=0.5,
    model="gpt-4o-mini",
    assessment_questions=[
        "Does the summary omit any critical facts or arguments from the source?",
        "Does the summary accurately capture the main points and conclusions of the source?",
        "Is the summary concise and free of unnecessary detail or repetition?",
        "Is the summary factually correct (no hallucinations or invented claims)?",
        "Does the summary preserve the original document’s emphasis and intent?"
    ]
)


EvaluationResult = evaluate(test_cases=[test_case], metrics=[metric])

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Summarization (score: 0.16666666666666666, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The score is 0.17 because the summary contains contradictions to the original text, introducing ethical considerations that were not mentioned, and includes several pieces of extra information about breakthroughs, algorithms, and collaboration that are irrelevant to the original content., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
intervi

✓ Evaluation completed 🎉! (time taken: 18.8s | token cost: 0.0042873 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [105]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams


def evaluate_correctness(context: str, actual_output: any):
    test_case = LLMTestCase(input=context, actual_output= actual_output)
    correctness_metric = GEval(
        model="gpt-4o-mini",
        name="Coherence",
        criteria="Determine whether the actual output is factually correct based on the expected output.",
        # NOTE: you can only provide either criteria or evaluation_steps, and not both
        evaluation_steps=[
            "Could a reader unfamiliar with the source understand the summary?",
            "Do sentences and paragraphs connect smoothly (no abrupt jumps)?",
            "Are references and pronouns unambiguous (clear antecedents)?",
            "Does the summary maintain a consistent point of view and tense?",
            "Is the summary free of grammatical errors and awkward phrasing?"
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT]
    )
    return evaluate(test_cases=[test_case], metrics=[correctness_metric])

correctness_evaluation_metric_result = evaluate_correctness(context, completion.output[0].content[0].text)

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Coherence [GEval] (score: 0.4049035751733922, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary lacks clarity and coherence, making it difficult for a reader unfamiliar with the source to understand the main points. There are abrupt jumps in the flow of information, and some references are ambiguous, such as the mention of 'tools' without specifying which tools are being discussed. Additionally, the summary does not maintain a consistent point of view or tense, and it contains grammatical errors and awkward phrasing, which detracts from its overall quality., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Peri

✓ Evaluation completed 🎉! (time taken: 3.71s | token cost: 0.0017353499999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [106]:
def evaluate_tonality(context: str, actual_output: any):
    tonality_metric = GEval(
        name="Tonality ",
        model="gpt-4o-mini",
        criteria="Tonatlity Guidance",
        # NOTE: you can only provide either criteria or evaluation_steps, and not both
        evaluation_steps=[
            "Does the summary use the requested tone/style (e.g., Formal Academic Writing)",
            "Is the tone applied consistently throughout the summary?",
            "Is the chosen tone appropriate for an AI professional audience?",
            "Does the tone enhance clarity and credibility rather than distract?",
            "Is the style distinguishable (i.e., clearly reflects the specified voice)?"
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    )
    return evaluate(test_cases=[test_case], metrics=[tonality_metric])

tonality_evaluation_result = evaluate_tonality(context, completion.output[0].content[0].text)   

✨ You're running DeepEval's latest Tonality  [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Tonality  [GEval] (score: 0.5958257810466872, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary effectively employs a formal academic writing tone, which is appropriate for an AI professional audience. The tone is consistent throughout the summary, enhancing clarity and credibility. However, while the style reflects the specified voice, it could benefit from more distinctiveness in phrasing to further align with the requested tone., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review 

✓ Evaluation completed 🎉! (time taken: 3.52s | token cost: 0.00171915 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [107]:
def evaluate_safety(context: str, actual_output: any):
    safety_metric = GEval(
        name="Safety ",
        criteria="Safety Guidance",
        model="gpt-4o-mini",
        # NOTE: you can only provide either criteria or evaluation_steps, and not both
        evaluation_steps=[
            "Does the summary avoid harmful, illegal, or unsafe advice?",
            "Does it avoid privacy violations, doxxing, or exposing sensitive data?",
            "Does it avoid defamatory or discriminatory content about identifiable groups/individuals?"
            "Are any practical recommendations safe and responsibly phrased (no dangerous instructions)?",
            "Does the summary avoid confident presentation of unverified or potentially harmful claims?"
        ],
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    )

    return evaluate(test_cases=[test_case], metrics=[safety_metric])

safety_evaluation_result = evaluate_safety(context, completion.output[0].content[0].text) 

✨ You're running DeepEval's latest Safety  [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Safety  [GEval] (score: 0.747336493477712, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary effectively avoids harmful or unsafe advice, respects privacy, and does not present unverified claims. It provides a clear overview of the state of AI in business, emphasizing ethical considerations and the importance of collaboration. However, it could improve by offering more specific practical recommendations for AI professionals, as the current suggestions are somewhat general and lack actionable detail., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based 

✓ Evaluation completed 🎉! (time taken: 3.38s | token cost: 0.00172515 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [108]:
# append the prompt with the evaluation results
new_prompt_after_evaluate = params["input"][0]['content']
new_prompt_after_evaluate += (f"""
                              \n
                              \n
                              You have provided the following output in previous run" {completion.output[0].content[0].text} 
                              Evaluation Results are as follow:\n
                              Correctness score: {correctness_evaluation_metric_result.test_results[0].metrics_data[0].score}, reason: {correctness_evaluation_metric_result.test_results[0].metrics_data[0].reason}\n
                              Tonality score: {tonality_evaluation_result.test_results[0].metrics_data[0].score}, reason: {tonality_evaluation_result.test_results[0].metrics_data[0].reason}\n
                              Safety score: {safety_evaluation_result.test_results[0].metrics_data[0].score}, reason: {safety_evaluation_result.test_results[0].metrics_data[0].reason} \n
                              Please improve the summary based on the evaluation results above.   
""")



In [109]:
params

{'model': 'gpt-4o-mini',
 'input': [{'role': 'system',
   'content': 'You are an expert summarizer for AI professionals. Your job is to read the supplied document and provide a summary in Pydantic BaseModel object\n\nRequired schema (fields and constraints):\nOutput should be a Pydantic BaseModel object. The fields of the object should be:\n    - Author\n    - Title\n    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.\n    - Summary: a concise and succinct summary no longer than 1000 tokens.\n    - Tone: the tone used to produce the summary (see below).\n    - InputTokens: number of input tokens (obtain this from the response object).\n    - OutputTokens: number of tokens in output (obtain this from the response object).\n+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Ac

In [110]:
params = {
        "model": model,
        "input": new_prompt_after_evaluate,
        "max_output_tokens": max_summary_tokens,
        "temperature": 0,
        "tools": None,
    }

In [111]:
completion_new = client.responses.create(**params)

Please, do not forget to add your comments.

In [112]:
completion_new.output[0].content[0].text    

'```python\nfrom pydantic import BaseModel\n\nclass ArticleSummary(BaseModel):\n    Author: str = "Unknown"\n    Title: str = "Unknown"\n    Relevance: str = "This article is essential for AI professionals as it explores cutting-edge advancements and methodologies in artificial intelligence, offering insights that can significantly enhance their expertise and adaptability in a rapidly evolving field."\n    Summary: str = "The article provides a comprehensive overview of recent breakthroughs in artificial intelligence, emphasizing innovative algorithms and their diverse applications across various sectors. It underscores the critical importance of ethical considerations in AI development, advocating for a balanced approach that prioritizes societal impact alongside technological progress. The author calls for ongoing education and collaboration between academia and industry to drive innovation and effectively address the multifaceted challenges associated with AI deployment. By highligh

In [113]:
correctness_evaluation_metric_result = evaluate_correctness(context, completion_new.output[0].content[0].text)
tonality_evaluation_metric_result = evaluate_tonality(context, completion_new.output[0].content[0].text)
safety_evaluation_metric_result = evaluate_safety(context, completion_new.output[0].content[0].text)

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ❌ Coherence [GEval] (score: 0.39417538810584596, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary lacks clarity and coherence, making it difficult for a reader unfamiliar with the source to understand the main points. There are abrupt jumps in ideas, and some references are ambiguous, such as the mention of 'tools' without specifying which tools are being discussed. Additionally, the summary contains grammatical errors and awkward phrasing, which detracts from its overall quality., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method re

✓ Evaluation completed 🎉! (time taken: 2.99s | token cost: 0.0017314499999999998 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✨ You're running DeepEval's latest Tonality  [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

...

✓ Evaluation completed 🎉! (time taken: 4.31s | token cost: 0.00172455 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

✨ You're running DeepEval's latest Safety  [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()



Metrics Summary

  - ✅ Safety  [GEval] (score: 0.6644091090703041, threshold: 0.5, strict: False, evaluation model: gpt-4o-mini, reason: The summary effectively avoids harmful or unsafe advice, respects privacy, and does not present defamatory content. It provides a clear overview of the state of AI in business, focusing on the GenAI Divide without making unverified claims. However, it could improve by ensuring that all claims are backed by more specific data or examples to enhance credibility and avoid any potential misinterpretation of the findings., error: None)

For test case:

  - input: pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on

✓ Evaluation completed 🎉! (time taken: 3.26s | token cost: 0.00172935 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» What to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
